# 01 函数逼近问题概览

第二章讨论的是插值：给定节点值，构造函数严格通过每个节点。第三章讨论更宽的任务：用一个简单函数空间去近似复杂函数或离散数据。

本节先区分三件事：

* **插值**：满足 $p(x_i)=y_i$；
* **函数逼近**：在某个函数空间中最小化 $\\|f-p\\|$；
* **曲线拟合**：对含噪声数据允许残差，整体上解释数据。

这三者都会画出一条曲线，但数学约束和适用场景不同。


## 三类问题的差别

插值适合查表值和精确采样。若数据来自实验测量，强制通过所有点可能会把噪声也插值进去。

函数逼近面对的是连续函数。例如给定 $f(x)=e^x\cos 3x$，希望在

$$
V_n=\operatorname{span}\{1,x,x^2,\dots,x^n\}
$$

或某个正交多项式空间中找到 $p_n$，使 $f-p_n$ 在某种范数下尽量小。

曲线拟合面对的是离散数据。常见模型是

$$
p_n(x)=c_0+c_1x+\cdots+c_nx^n,
$$

并最小化残差平方和

$$
\sum_i (y_i-p_n(x_i))^2.
$$


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import polynomial_eval, polynomial_least_squares

rng = np.random.default_rng(2026)
x_data = np.linspace(-1.0, 1.0, 18)
y_true = np.sin(2 * np.pi * x_data)
y_noisy = y_true + 0.12 * rng.normal(size=x_data.size)

x_eval = np.linspace(-1.0, 1.0, 400)
coefficients = polynomial_least_squares(x_data, y_noisy, degree=5)
y_fit = polynomial_eval(coefficients, x_eval)

plt.figure(figsize=(8, 4.5))
plt.scatter(x_data, y_noisy, color="black", label="含噪声数据")
plt.plot(x_eval, np.sin(2 * np.pi * x_eval), color="gray", label="真实函数")
plt.plot(x_eval, y_fit, label="五次最小二乘拟合")
plt.title("拟合允许残差，而不是强制通过每个点")
plt.xlabel("x")
plt.ylabel("函数值")
plt.legend()
plt.tight_layout()
plt.show()


## 误差范数

逼近问题必须先说明“接近”的意义。常见选择有三种。

一致范数：

$$
\\|f-g\\|_\infty=\max_{x\in[a,b]} |f(x)-g(x)|.
$$

平方范数：

$$
\\|f-g\\|_2=\left(\int_a^b |f(x)-g(x)|^2 w(x)\,dx\right)^{1/2}.
$$

离散二范数：

$$
\\|r\\|_2=\left(\sum_{i=1}^{m}r_i^2\right)^{1/2}.
$$

一致范数关注最坏点误差，平方范数关注平均意义，离散二范数对应最小二乘拟合。


In [ ]:
def infinity_norm_error(f, g, x_grid):
    return np.max(np.abs(f(x_grid) - g(x_grid)))

# 教学式离散二范数：先构造残差，再取平方和开方。
residual = y_noisy - polynomial_eval(coefficients, x_data)
discrete_l2 = np.sqrt(np.sum(residual**2))

print(f"离散残差二范数: {discrete_l2:.3e}")
print(f"拟合多项式系数（升幂）: {coefficients}")


## 本章阅读顺序

本章后续内容按以下顺序展开：

1. 正交多项式与函数逼近：Chebyshev 和 Legendre；
2. 最小二乘曲线拟合：离散数据、Vandermonde 矩阵、过拟合；
3. Padé 有理逼近：从 Taylor 系数构造有理函数；
4. Fourier/FFT、自适应逼近和最佳一致逼近作为后续扩展框架。

这一章和第二章的关系是：Chebyshev 节点仍然出现，但重点不再是“插值公式”，而是“函数逼近工具”。


## 小结

* 插值要求精确通过节点；逼近要求在函数空间中整体接近；拟合允许离散残差。
* 误差范数决定“最好”的含义。
* 正交多项式把最佳平方逼近变成投影问题。
* 最小二乘拟合是离散数据上的投影问题。
* Padé 逼近用有理函数改善 Taylor 多项式的局限。
